# Generación continua de logs — Archivo CSV (`logs_servidor_v2.csv`)

Este notebook simula un flujo **near real-time**: cada 2-4 segundos agrega una línea nueva a un archivo CSV que crece de forma continua. Logstash, con el input `file` en modo tail (`sincedb_path`), detecta las líneas nuevas y las indexa en Elasticsearch a través del filtro `csv`.



**Requisitos:**
```
pip install Faker
```

In [1]:
import csv
import time
import random
import os
from datetime import datetime
from faker import Faker

fake = Faker()

In [2]:
ARCHIVO_CSV = "E:/USFQ/logstash-7.17.10/data/logs_servidor.csv"
COLUMNAS = ["log_id", "appid", "host", "ip_origen", "estado_servidor", "ping_ms", "fecha_log"]

APPIDS_MUESTRA = [730, 578080, 570, 271590, 440, 105600, 252490, 4000, 1091500, 292030]
ESTADOS = ["ONLINE", "ONLINE", "ONLINE", "ONLINE", "WARNING", "ERROR", "TIMEOUT"]

## Generador de registros de log

Función pura: no depende de variables globales de escritura (sí usa las constantes de dominio de arriba, igual que `generar_evento()` en el notebook de MySQL). Devuelve un **diccionario** con claves iguales a `COLUMNAS`, listo para escribirse con `csv.DictWriter`.

In [3]:
def generar_log():
    estado = random.choice(ESTADOS)
    ping = random.randint(10, 80) if estado == "ONLINE" else random.randint(300, 999)

    return {
        "log_id": int(time.time() * 1000),
        "appid": random.choice(APPIDS_MUESTRA),
        "host": fake.hostname(),
        "ip_origen": fake.ipv4(),
        "estado_servidor": estado,
        "ping_ms": ping,
        "fecha_log": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

## Función de escritura

Recibe el diccionario y lo escribe con `DictWriter`, igual espíritu que `insertar_evento(cursor, evento)`: el registro conserva sus nombres de campo hasta el final, en vez de convertirse en una lista posicional.

In [4]:
def escribir_log(writer, registro, archivo_handle):
    writer.writerow(registro)
    archivo_handle.flush()

## Bucle de simulación (near real-time)

Agrega una línea nueva cada 2-4 segundos. Detén la celda (interrumpir kernel) para parar la simulación.

In [ ]:
print(f"Iniciando simulación de logs de servidor en: {ARCHIVO_CSV}")

os.makedirs(os.path.dirname(ARCHIVO_CSV), exist_ok=True)
es_nuevo = not os.path.exists(ARCHIVO_CSV) or os.stat(ARCHIVO_CSV).st_size == 0

with open(ARCHIVO_CSV, mode="a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=COLUMNAS)

    if es_nuevo:
        writer.writeheader()
        f.flush()
        print("Archivo nuevo creado con encabezados.")

    contador = 0
    try:
        print("Insertando logs cada 2 a 4 segundos. Interrumpe el kernel para detener.\n")
        while True:
            registro = generar_log()
            escribir_log(writer, registro, f)
            contador += 1
            print(f"[{contador}] Log añadido: {registro}")
            time.sleep(random.randint(2, 4))
    except KeyboardInterrupt:
        print("\nSimulación detenida por el usuario.")
        print(f"Total de logs generados en esta sesión: {contador}")

Iniciando simulación de logs de servidor en: E:/USFQ/logstash-7.17.10/data/logs_servidor.csv
Insertando logs cada 2 a 4 segundos. Interrumpe el kernel para detener.

[1] Log añadido: {'log_id': 1788671991501, 'appid': 271590, 'host': 'db-29.beck.info', 'ip_origen': '187.105.231.98', 'estado_servidor': 'ONLINE', 'ping_ms': 24, 'fecha_log': '2026-09-06 00:19:51'}
[2] Log añadido: {'log_id': 1788671995506, 'appid': 252490, 'host': 'laptop-48.russell.org', 'ip_origen': '10.109.68.15', 'estado_servidor': 'ONLINE', 'ping_ms': 23, 'fecha_log': '2026-09-06 00:19:55'}
[3] Log añadido: {'log_id': 1788671998507, 'appid': 578080, 'host': 'lt-11.jones.org', 'ip_origen': '53.63.51.25', 'estado_servidor': 'WARNING', 'ping_ms': 843, 'fecha_log': '2026-09-06 00:19:58'}
[4] Log añadido: {'log_id': 1788672001509, 'appid': 252490, 'host': 'desktop-93.sanders.com', 'ip_origen': '68.104.103.165', 'estado_servidor': 'WARNING', 'ping_ms': 725, 'fecha_log': '2026-09-06 00:20:01'}
[5] Log añadido: {'log_id': 17